# CSE598/EEE589 Project 05 — distributed experiments

This notebook repeats setup, then launches distributed workers only when at least two NVIDIA GPUs are visible to this runtime. Workers are external `torchrun` processes. They are not started inside the notebook kernel. Two ordinary Colab sessions are not a 100 Gb Ethernet cluster.

If the gate fails, the notebook prints the blocker and the server commands. It does not fabricate ranks, bandwidth, or overlap.


## Configuration


In [ ]:
# These values clone the project on a fresh Colab runtime.
# Leave ZIP_PATH and PROJECT_DIR empty.
REPO_URL = "https://github.com/nazndev/long-context-attention-benchmark.git"
REPO_COMMIT = "6befb712fb4666afbf8961f484f24e0cf4c9511d"
ZIP_PATH = ""
PROJECT_DIR = ""


## Setup


In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

def discover(start):
    root = Path(start).resolve()
    pool = [root, *root.parents]
    if root.is_dir():
        for child in root.iterdir():
            if child.is_dir() and child.name not in {".venv", ".git"}:
                pool.append(child)
                for grandchild in child.iterdir():
                    if grandchild.is_dir() and grandchild.name not in {".venv", ".git"}:
                        pool.append(grandchild)
    for candidate in pool:
        if (candidate / "src" / "spbench" / "__init__.py").is_file():
            return candidate
    return None

if ZIP_PATH:
    dest = Path("/content/long-context-attention-benchmark" if os.path.isdir("/content") else "colab_unzipped")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(dest)
    project = discover(dest)
elif REPO_URL:
    if not REPO_COMMIT:
        raise ValueError("Set REPO_COMMIT to a full commit hash. Do not track a moving branch.")
    dest = Path("/content/long-context-attention-benchmark" if os.path.isdir("/content") else "colab_clone")
    if not (dest / ".git").exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(dest)])
    subprocess.check_call(["git", "checkout", REPO_COMMIT], cwd=dest)
    project = discover(dest)
else:
    project = Path(PROJECT_DIR).resolve() if PROJECT_DIR else discover(Path.cwd())

if project is None:
    raise FileNotFoundError(
        "No project source found. On a fresh Colab runtime, upload the project ZIP and set ZIP_PATH, "
        "or set REPO_URL and REPO_COMMIT."
    )
os.chdir(project)
src = project / "src"
for entry in (src, project):
    entry_text = str(entry)
    if entry_text not in sys.path:
        sys.path.insert(0, entry_text)
print("Project directory:", project)


In [ ]:
import importlib.util
import sys
bootstrap_path = project / "scripts" / "colab_bootstrap.py"
spec = importlib.util.spec_from_file_location("colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
bootstrap.install_project(project)
src = project / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))
import spbench
print("spbench", spbench.__version__)


## GPU-count gate


In [ ]:
import json
from spbench.dependencies import probe_dependencies
from spbench.hardware import detect_hardware
hardware = detect_hardware()
dependencies = probe_dependencies()
gpu_count = int(hardware["cuda_device_count"])
print(json.dumps({"cuda_device_count": gpu_count, "gpu_names": hardware["gpu_names"], "dependencies": dependencies}, indent=2))
launch = gpu_count >= 2 and dependencies.get("ring_flash_attn", False)
print("NCCL launch permitted for Ring:" , bool(launch))
print("True USP (both degrees greater than 1) needs at least 4 visible GPUs. Visible:", gpu_count)
if gpu_count < 2:
    print("BLOCKER: fewer than 2 CUDA devices.")
    print("Single-node server command, after nvidia-smi and dependency installation:")
    print("torchrun --standalone --nproc-per-node=2 -m spbench.benchmark --method ring --seq-len 4096 --phase forward_backward")
    print("Two-node server command, after the 100 GbE interface is verified:")
    print("torchrun --nnodes=2 --nproc-per-node=2 --node-rank=0 --master-addr=<approved_node0_ip> --master-port=<approved_port> -m spbench.benchmark --method usp --ulysses-degree 2 --ring-degree 2 --seq-len 4096 --phase forward_backward")
    print("No distributed worker was started.")


## External torchrun, only when the gate is open


In [ ]:
import subprocess
import sys
if gpu_count < 2:
    print("Skipped torchrun. The gate above explains why.")
else:
    command = [
        sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc-per-node={min(gpu_count, 2)}",
        "-m", "spbench.benchmark", "--method", "ring", "--variant", "basic", "--seq-len", "128", "--phase", "forward",
        "--output", str(project / "results" / "raw" / "ring_smoke.jsonl"),
    ]
    completed = subprocess.run(command, cwd=project, text=True, capture_output=True, timeout=180)
    print(completed.stdout)
    print(completed.stderr)
    print("return code", completed.returncode)
    if completed.returncode != 0:
        print("The worker failed. Its status is a failure, not a throughput result.")
